In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

144

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

print(result)

questions=['I just found this course — is it still okay to enroll now, or is it too late?', 'If I join late, can I still get a certificate, or did I miss the chance?', 'What’s the deadline for the final project if I want the certificate?', 'Do I need to submit the project before the course stops accepting submissions to qualify for the certificate?', 'Can someone who discovers the course after it starts still complete everything and get certified?']


In [12]:
print(result.questions)

['I just found this course — is it still okay to enroll now, or is it too late?', 'If I join late, can I still get a certificate, or did I miss the chance?', 'What’s the deadline for the final project if I want the certificate?', 'Do I need to submit the project before the course stops accepting submissions to qualify for the certificate?', 'Can someone who discovers the course after it starts still complete everything and get certified?']


In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course late — can I still sign up and take part?', 'Am I allowed to join after the course has already started?', 'If I enroll now, is it still possible to get the certificate, or am I too late?', 'What’s the deadline for submitting the project if I want a certificate?', 'Can I still receive course completion proof if I join after submissions close?']


In [15]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=92, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=299)

In [17]:
from evaluation_utils import calc_price

In [18]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.000414, 'total_cost': 0.00056925}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — can I still sign up and take part?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to join after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I enroll now, is it still possible to get the certificate, or am I too late?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for submitting the project if I want a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I still receive course completion proof if I join after submissions close?',
  'document': '74eb249bbf'}]

In [20]:
import pandas as pd

In [21]:
pd.DataFrame(records)

,question,document
0,I just found this course late — can I still si...,74eb249bbf
1,Am I allowed to join after the course has alre...,74eb249bbf
2,"If I enroll now, is it still possible to get t...",74eb249bbf
3,What’s the deadline for submitting the project...,74eb249bbf
4,Can I still receive course completion proof if...,74eb249bbf


In [22]:
from evaluation_utils import llm_structured_retry

In [23]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [24]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [26]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [27]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/144 [00:00<?, ?it/s]

In [28]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

720

In [29]:
ground_truth[20]

{'question': 'How do I find my entry on the leaderboard if I can’t spot my name right away?',
 'document': 'c2903069a0'}

In [30]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.11301975000000003

In [31]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.11301975000000003

In [32]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [34]:
df_ground_truth.to_csv("ground_truth-new.csv", index=False)